<div class="jupyter-biolm-header">
    <img style="float: left; padding-right: 10px; height: 60px" src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/logo.png">
    <p>
    <br>
    <br>
    <br>
    </p>
</div>

# Screen 1,000 Peptides Before Lunch

Multi-stage pipeline to screen a peptide library for thermal stability and solubility.

<br>

<table class="jupyter-biolm-header-table" style="width: 100%; border-collapse: collapse; background-color: white; float: left;">
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://www.svgrepo.com/show/354202/postman-icon.svg" style="height: 15px; float: left; padding-right: 10px;"><a href="https://api.biolm.ai/">  <h5 style="margin: 0;"><b>Postman API Docs</b></h5></a>
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c3/Python-logo-notext.svg/1869px-Python-logo-notext.svg.png" style="height: 15px; float: left; padding-right: 10px;"><a href="https://docs.biolm.ai/en/latest/index.html"><h5 style="margin: 0;"><b>Python SDK Docs</b></h5></a>
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
        </td>
    </tr>
</table>

<br>

---

> **⚠️ Preview Feature** — The `biolmai.pipeline` module used in this guide is currently in preview and not yet publicly released. Access is available to early users on request. [Contact us](https://biolm.ai) to get access.

**What you'll learn:**
- Defining a multi-stage pipeline with parallel predictions
- Filtering by melting temperature and solubility
- Exploring results with `summary()`, `stats()`, and SQL queries

**Requirements:**
```
pip install biolmai[pipeline] matplotlib
export BIOLMAI_TOKEN=your-token-here
```

## Setup

In [ ]:
import os
from biolmai.pipeline import (
    DataPipeline, DuckDBDataStore,
    ThresholdFilter, RankingFilter,
    ValidAminoAcidFilter, EmbeddingSpec,
    DiversitySamplingFilter,
)

TOKEN = os.environ.get("BIOLMAI_TOKEN", "")
if not TOKEN:
    raise EnvironmentError(
        "Set BIOLMAI_TOKEN before running.\n"
        "Get one at https://biolm.ai/ui/accounts/user-api-tokens/"
    )

## Peptide library

30 antimicrobial peptides of varying length, charge, and hydrophobicity.

In [ ]:
MY_PEPTIDES = [
    # Magainins / frog-derived
    "GIGKFLHSAKKFGKAFVGEIMNS",
    "GIGKFLHSAGKFGKAFVGEIMKS",
    "GLFDIIKKIAESF",
    "GLFDIVKKVVGALGSL",
    "FLPLILRKIVTAL",
    # Human defensins / cathelicidins
    "LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES",
    "RLFDKIRQVIRKF",
    "KWKLFKKIPKFLHLAKKF",
    # Insect-derived
    "GIGAVLKVLTTGLPALISWIKRKRQQ",
    "VDKGSYLPRPTPPRPIYNRN",
    # Synthetic / designed
    "KLAKLAKKLAKLAK",
    "LKLLKKLLKLLKKL",
    "RRWWRRWWRR",
    "KWKWKWKWKW",
    "GIKKFLGSIWKFIKAFVKEIMN",
    # Short peptides
    "RRWQWR",
    "RWRWRW",
    "FKRIVQRIKDFL",
    "KFLKKAKKFGK",
    "GIGKFLHSAK",
    "KWKLFKKI",
    "RLFDKIRQ",
    # Longer peptides
    "GLFDIIKKIAESFLPKV",
    "GIGKFLHSAKKFGKAFV",
    "KWKLFKKIPKFLHLAK",
]
print(f"{len(MY_PEPTIDES)} peptides, length range: {min(len(s) for s in MY_PEPTIDES)}–{max(len(s) for s in MY_PEPTIDES)} aa")

## Build and run the pipeline

The dependency graph:
```
validate
   ├── predict_tm   ─┐
   └── predict_sol  ─┴── filter_tm >= 40°C ── rank top 15 by solubility
```
Both prediction stages run in **parallel** because they share the same dependency.

In [ ]:
pipeline = DataPipeline(sequences=MY_PEPTIDES, verbose=True)

pipeline.add_filter(ValidAminoAcidFilter(), stage_name="validate")

pipeline.add_prediction(
    "temperature-regression", extractions="prediction",
    columns="melting_temperature", stage_name="predict_tm",
    depends_on=["validate"],
)
pipeline.add_prediction(
    "biolmsol", extractions="solubility_score",
    columns="solubility", stage_name="predict_sol",
    depends_on=["validate"],
)

pipeline.add_filter(ThresholdFilter("melting_temperature", min_value=40.0), stage_name="filter_tm")
pipeline.add_filter(RankingFilter("solubility", n=15, ascending=False), stage_name="top15")

pipeline.run()

## Explore results

In [ ]:
pipeline.summary()

In [ ]:
pipeline.stats()

In [ ]:
# Top 10 sequences by melting temperature
pipeline.query("""
    SELECT s.sequence,
           MAX(CASE WHEN p.prediction_type = 'melting_temperature' THEN p.value END) AS tm,
           MAX(CASE WHEN p.prediction_type = 'solubility' THEN p.value END) AS solubility
    FROM sequences s
    JOIN predictions p ON s.sequence_id = p.sequence_id
    GROUP BY s.sequence
    ORDER BY tm DESC
    LIMIT 10
""")

In [ ]:
pipeline.plot("funnel")

## Next Steps

Check out additional tutorials at [jupyter.biolm.ai](https://jupyter.biolm.ai),
or head over to our [BioLM Documentation](https://docs.biolm.ai) to explore
additional models and functionality.

#### See more use-cases and APIs on your [BioLM Console Catalog](https://biolm.ai/console/catalog/).
<br>

##### BioLM hosts deep learning models and runs inference at scale. You do the science.
<br>

<table class="jupyter-biolm-header-table" style="width: 100%; border-collapse: collapse; background-color: white; float: left;">
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/enzyme_engineering_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Enzyme Engineering
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/antibody_engineering_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Antibody Engineering
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/biosecurity_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Biosecurity
        </td>
    </tr>
    <tr>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/single_cell_genomics_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Single-Cell Genomics
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/dna_seq_modeling_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> DNA Sequence Modelling
        </td>
        <td style="text-align: left; vertical-align: middle; background-color: white;">
            <img src="https://d31e6ufxekikrt.cloudfront.net/static/ui/images/console-overview/finetuning_icon.png"  style="height: 40px; float: left; padding-right: 10px;"> Finetuning
        </td>
    </tr>
</table>

#### [**Contact us**](https://biolm.ai/ui/contact-us/) to learn more.